In [ ]:
import json
import google.generativeai as genai
from time import sleep
import re

# === Cấu hình API Key Gemini ===
genai.configure(api_key="")

input_file = "/kaggle/input/5x1000data/qwen_finetune_data3.jsonl"
output_file = "/kaggle/working/qwen_finetune_data_modified3.jsonl"
log_file = "/kaggle/working/process_log3.txt"
batch_size = 1
temperature = 0.3
max_retry = 3

# === DANH SÁCH MODEL FALLBACK THEO THỨ TỰ ƯU TIÊN ===
model_priority_list = [
    'gemini-2.5-flash-lite',
    'gemini-2.5-pro',
    'gemini-2.5-flash',
    'gemini-2.0-flash',
    'gemini-2.0-flash-exp',
    'gemini-2.5-flash-tts'
]

max_output_tokens = 8192

def get_available_model():
    """Lấy model đầu tiên khả dụng trong danh sách"""
    for model_name in model_priority_list:
        try:
            model = genai.GenerativeModel(model_name)
            print(f"✅ Sử dụng model: {model_name}")
            return model_name
        except Exception as e:
            print(f"❌ Model {model_name} không khả dụng: {e}")
            continue
    print(f"⚠️  Cảnh báo: Không có model nào khả dụng, thử dùng {model_priority_list[0]}")
    return model_priority_list[0]

def clean_json_response(text):
    """Làm sạch response JSON từ API"""
    cleaned = re.sub(r'```json\s*', '', text)
    cleaned = re.sub(r'\s*```', '', cleaned)
    cleaned = cleaned.strip()
    
    cleaned = re.sub(r'^[^{[]*', '', cleaned)
    cleaned = re.sub(r'[^}\]]*$', '', cleaned)
    
    return cleaned

def repair_truncated_json(json_str):
    """Sửa JSON bị cắt ngang"""
    if not json_str.strip():
        return json_str
        
    open_braces = json_str.count('{')
    close_braces = json_str.count('}')
    open_brackets = json_str.count('[')
    close_brackets = json_str.count(']')
    
    repaired = json_str
    
    if open_braces > close_braces:
        repaired += '}' * (open_braces - close_braces)
    
    if open_brackets > close_brackets:
        repaired += ']' * (open_brackets - close_brackets)
    
    return repaired

def extract_and_parse_json(response_text):
    """Trích xuất và parse JSON từ response text"""
    print(f"🔧 Đang xử lý response dài {len(response_text)} chars...")
    
    cleaned_text = clean_json_response(response_text)
    print(f"🔧 Sau khi làm sạch: {len(cleaned_text)} chars")
    
    # Thử parse trực tiếp
    try:
        json_data = json.loads(cleaned_text)
        print("✅ Parse trực tiếp thành công")
        return json_data
    except json.JSONDecodeError as e:
        print(f"❌ Parse trực tiếp thất bại: {e}")
    
    # Tìm JSON array bằng regex
    array_pattern = r'\[\s*\{[\s\S]*?\}\s*\]'
    array_matches = re.findall(array_pattern, cleaned_text, re.DOTALL)
    
    if array_matches:
        print(f"✅ Tìm thấy {len(array_matches)} JSON arrays bằng regex")
        json_str = max(array_matches, key=len)
        json_str_repaired = repair_truncated_json(json_str)
        print(f"🔧 Đã sửa JSON (từ {len(json_str)} lên {len(json_str_repaired)} chars)")
        
        try:
            json_data = json.loads(json_str_repaired)
            print("✅ Parse JSON đã sửa thành công")
            return json_data
        except json.JSONDecodeError as e2:
            print(f"❌ Vẫn lỗi JSON sau khi sửa: {e2}")
    
    # Tìm thủ công
    start_idx = cleaned_text.find('[')
    end_idx = cleaned_text.rfind(']')
    
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        json_str = cleaned_text[start_idx:end_idx+1]
        print(f"✅ Tìm thấy JSON bằng manual extraction ({len(json_str)} chars)")
        json_str_repaired = repair_truncated_json(json_str)
        
        try:
            json_data = json.loads(json_str_repaired)
            print("✅ Parse JSON manual thành công")
            return json_data
        except json.JSONDecodeError as e3:
            print(f"❌ Lỗi parse JSON manual: {e3}")
    
    raise ValueError("Không thể trích xuất JSON từ response")

def enhance_conversations_batch(batch):
    """
    Dựa trên conversations gốc, tạo conversations mới xoay quanh cùng chủ đề
    nhưng với câu hỏi và câu trả lời được mở rộng, kiểm chứng thông tin
    """
    # Trích xuất chủ đề từ batch
    topics = []
    for data in batch:
        user_messages = [msg["content"] for msg in data["messages"] if msg["role"] == "user"]
        if user_messages:
            # Lấy câu hỏi đầu tiên làm đại diện cho chủ đề
            first_question = user_messages[0]
            # Giới hạn độ dài để tránh prompt quá dài
            topic = first_question[:200] + "..." if len(first_question) > 200 else first_question
            topics.append(topic)
        else:
            topics.append("Lịch sử Việt Nam")
    
    prompt = f"""
DỰA TRÊN CÁC CHỦ ĐỀ HỘI THOẠI GỐC SAU, HÃY TẠO {len(batch)} HỘI THOẠI MỚI VỚI THÔNG TIN ĐƯỢC KIỂM CHỨNG VÀ MỞ RỘNG:

CHỦ ĐỀ GỐC:
{chr(10).join([f"{i+1}. {topic}" for i, topic in enumerate(topics)])}

YÊU CẦU:
1. TẠO CÂU HỎI VÀ CÂU TRẢ LỜI MỚI XOAY QUANH CHỦ ĐỀ GỐC
2. KIỂM CHỨNG THÔNG TIN LỊCH SỬ - PHẢI CHÍNH XÁC TUYỆT ĐỐI
3. BỔ SUNG THÔNG TIN CHI TIẾT: ngày tháng, nhân vật, địa điểm, nguyên nhân, kết quả
4. Mỗi hội thoại có 3-4 cặp Q&A chất lượng cao
5. Giữ nguyên tinh thần và chủ đề của hội thoại gốc

CÁC LOẠI CÂU HỎI NÊN TẠO:
- Câu hỏi phân tích nguyên nhân - hệ quả
- Câu hỏi về nhân vật và vai trò lịch sử
- Câu hỏi so sánh với sự kiện khác
- Câu hỏi về ý nghĩa và bài học lịch sử

ĐỊNH DẠNG JSON:
[
  {{
    "messages": [
      {{"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."}},
      {{"role": "user", "content": "Câu hỏi phân tích về chủ đề..."}},
      {{"role": "assistant", "content": "Trả lời chi tiết với thông tin đã kiểm chứng..."}}
    ]
  }}
]

QUAN TRỌNG: CHỈ TRẢ VỀ JSON, KHÔNG THÊM VĂN BẢN NÀO KHÁC.
"""

    current_model = get_available_model()
    
    for attempt in range(1, max_retry + 1):
        try:
            print(f"🔄 Attempt {attempt} với model {current_model}...")
            
            model = genai.GenerativeModel(current_model)
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=temperature,
                    max_output_tokens=max_output_tokens,
                    top_p=0.8
                )
            )
            
            response_text = response.text.strip()
            print(f"📄 Raw response length: {len(response_text)} chars")
            
            if len(response_text) > 300:
                print(f"📄 First 300 chars: {response_text[:300]}...")
            
            # Xử lý JSON
            json_data = extract_and_parse_json(response_text)
            
            # Kiểm tra cấu trúc
            if not isinstance(json_data, list):
                raise ValueError("Kết quả không phải là list")
                
            print(f"✅ Đã tạo được {len(json_data)} conversations")
                
            for i, item in enumerate(json_data):
                if "messages" not in item:
                    raise ValueError(f"Thiếu key 'messages' trong item {i}")
                if not isinstance(item["messages"], list):
                    raise ValueError(f"'messages' trong item {i} không phải là list")
                if len(item["messages"]) < 3:
                    raise ValueError(f"Hội thoại {i} quá ngắn")
                    
            return [item["messages"] for item in json_data]
            
        except Exception as e:
            print(f"❌ Lỗi attempt {attempt} với model {current_model}: {str(e)[:200]}")
            
            if attempt < max_retry:
                next_model_index = (model_priority_list.index(current_model) + 1) % len(model_priority_list)
                current_model = model_priority_list[next_model_index]
                print(f"🔄 Chuyển sang model: {current_model}")
                sleep(3)
            else:
                print(f"💥 Đã thử tất cả {max_retry} lần")
                return []

def create_fallback_conversations(batch):
    """Tạo hội thoại fallback dựa trên batch gốc"""
    fallback_data = []
    
    for data in batch:
        # Tìm chủ đề từ conversation gốc
        user_messages = [msg["content"] for msg in data["messages"] if msg["role"] == "user"]
        topic = user_messages[0] if user_messages else "Lịch sử Việt Nam"
        
        # Tạo conversation mới đơn giản
        messages = [
            {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
            {"role": "user", "content": f"Hãy phân tích về {topic}"},
            {"role": "assistant", "content": f"Thông tin về {topic} dựa trên các tài liệu lịch sử đã được kiểm chứng."},
            {"role": "user", "content": f"Ai là nhân vật chính liên quan đến {topic}?"},
            {"role": "assistant", "content": f"Các nhân vật lịch sử quan trọng trong sự kiện {topic}."},
            {"role": "user", "content": f"Ý nghĩa lịch sử của {topic}?"},
            {"role": "assistant", "content": f"{topic} có ý nghĩa quan trọng trong tiến trình lịch sử Việt Nam."}
        ]
        
        fallback_data.append(messages)
    
    return fallback_data

# === XỬ LÝ CHÍNH - ĐỌC TỪ FILE INPUT ===
processed_count = 0
batch = []

print(f"🎯 Bắt đầu xử lý file: {input_file}")

with open(input_file, "r", encoding="utf-8") as f_in, \
     open(output_file, "w", encoding="utf-8") as f_out, \
     open(log_file, "w", encoding="utf-8") as f_log:

    # Đếm tổng số dòng
    f_in.seek(0)
    total_lines = sum(1 for line in f_in)
    f_in.seek(0)
    
    print(f"📊 Tổng số conversations trong file: {total_lines}")

    for line_idx, line in enumerate(f_in, start=1):
        try:
            data = json.loads(line)
            if "messages" not in data:
                print(f"⚠️ Dòng {line_idx}: thiếu key 'messages'. Bỏ qua.")
                continue
            batch.append(data)
        except json.JSONDecodeError as e:
            print(f"⚠️ Lỗi JSON dòng {line_idx}: {e}. Bỏ qua.")
            continue
            
        # Xử lý batch khi đủ số lượng hoặc là dòng cuối
        if len(batch) >= batch_size or line_idx == total_lines:
            if not batch:
                continue
                
            print(f"\n🚀 Đang xử lý batch {processed_count + 1}-{processed_count + len(batch)}...")
            
            # Gọi API để tạo conversations mới dựa trên batch gốc
            enhanced_conversations = enhance_conversations_batch(batch)
            
            if not enhanced_conversations:
                print("🔄 Sử dụng fallback...")
                enhanced_conversations = create_fallback_conversations(batch)
            
            # Ghi kết quả
            for messages in enhanced_conversations:
                output_data = {"messages": messages}
                f_out.write(json.dumps(output_data, ensure_ascii=False) + "\n")
                processed_count += 1
            
            f_log.write(f"Đã xử lý {processed_count}/{total_lines} (đến dòng {line_idx})\n")
            print(f"✅ Đã xử lý {processed_count}/{total_lines} conversations")
            
            batch = []
            sleep(5)  # Chờ giữa các batch

print(f"\n🎉 Hoàn tất! Đã tạo {processed_count} conversations mới dựa trên dữ liệu gốc.")
print(f"📁 Output: {output_file}")
print(f"📋 Log: {log_file}")

🎯 Bắt đầu xử lý file: /kaggle/input/5x1000data/qwen_finetune_data3.jsonl
📊 Tổng số conversations trong file: 1000

🚀 Đang xử lý batch 1-1...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 14101 chars
📄 First 300 chars: ```json
[
  {
    "messages": [
      {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
      {
        "role": "user",
        "content": "Việc UNESCO công nhận Quan họ là Di sản văn hóa phi vật thể đại diện của nhân loại vào ngày 30 tháng 9 năm 2009 tại Abu Dhabi, Các Tiểu vươn...
🔧 Đang xử lý response dài 14101 chars...
🔧 Sau khi làm sạch: 14089 chars
✅ Parse trực tiếp thành công
✅ Đã tạo được 4 conversations
✅ Đã xử lý 4/1000 conversations

🚀 Đang xử lý batch 5-5...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 6580 chars
📄 First 300 chars: ```json
[
  {
    "messages": [
      {"role": "system", "content": "Bạn là chuyên gi